# File I/O and Error Handling in Python

## Introduction
Scientific programs often need to read input data from files and write results to external files. This is known as **file input/output** (file I/O). In addition, a program should be able to handle unexpected situations, such as missing files or invalid input values, without stopping in an uncontrolled way.

In Python, file objects and context managers are used to open, read, write, and close files safely, while exceptions provide a structured way to detect and handle runtime errors.

## 1. Opening and Closing Files

In Python, the basic syntax for opening a file is:

```python
file = open("filename", "mode")
```

The `filename` is the name of the file, while the `mode` specifies the operation to perform. Common modes are:

| Mode | Meaning |
| --- | --- |
| `"r"` | open for reading; this is the default mode |
| `"r+"` | open for both reading and writing |
| `"w"` | open for writing; overwrite the file if it exists, create it if it does not exist |
| `"a"` | open for appending; write new content at the end of the file |
| `"a+"` | open for reading and appending; create the file if it does not exist |

After a file operation is completed, the file should be closed using:

```python
file.close()
```

In practice, the recommended style is to use the `with` statement. It automatically closes the file when the block is completed, even if an error occurs inside the block.

In [1]:
# Create a small input file for the examples in this notebook.
with open("input.txt", "w") as file:
    file.write("10.0\n")
    file.write("20.0\n")
    file.write("30.0\n")

print("input.txt created")

input.txt created


## 2. Reading From a File

Once a file is opened in read mode, several methods can be used:

* `read()` reads the entire file as a single string;
* `readline()` reads one line at a time;
* `readlines()` reads all lines and returns them as a list of strings.

In [2]:
# Read the entire file.
with open("input.txt", "r") as file:
    content = file.read()

print(content)

10.0
20.0
30.0



In [3]:
# Read all lines and convert them to floating-point numbers.
with open("input.txt", "r") as file:
    lines = file.readlines()

values = [float(line) for line in lines]

print("Raw lines:", lines)
print("Numerical values:", values)

Raw lines: ['10.0\n', '20.0\n', '30.0\n']
Numerical values: [10.0, 20.0, 30.0]


The conversion from strings to numbers is a common step in scientific scripts. If the file contains invalid values, this conversion can raise an exception. This will be discussed in the error-handling section.

## 3. Writing and Appending to a File

To write data to a file, the file can be opened in write mode (`"w"`). This mode creates a new file or overwrites an existing one.

The method `write()` writes a string to the file. The method `writelines()` writes a sequence of strings to the same file.

In [4]:
# Write mode: create or overwrite a file.
with open("output.txt", "w") as file:
    file.write("Results\n")
    file.write("-------\n")

# Append mode: add new content at the end of the file.
with open("output.txt", "a") as file:
    file.write("Mean value: 20.0\n")

with open("output.txt", "r") as file:
    print(file.read())

Results
-------
Mean value: 20.0



In [5]:
# writelines writes multiple strings to the same file.
lines_to_write = [
    "temperature pressure\n",
    "300.0 1.0\n",
    "400.0 1.0\n",
]

with open("table.txt", "w") as file:
    file.writelines(lines_to_write)

with open("table.txt", "r") as file:
    print(file.read())

temperature pressure
300.0 1.0
400.0 1.0



## 4. Error Handling with Exceptions

An exception is an error detected during program execution. If an exception is not handled, the program stops and displays an error message.

The basic structure for handling exceptions is:

```python
try:
    # code that may raise an exception
except SomeException:
    # code executed if that exception occurs
```

This is useful when reading external input, because files may be missing or may contain values in an unexpected format.

In [6]:
raw_value = "not_a_number"

try:
    value = float(raw_value)
    print(f"Converted value: {value}")
except ValueError:
    print("Invalid input: a numerical value is required.")

Invalid input: a numerical value is required.


Multiple exceptions can be handled separately. The optional `else` block is executed only if the `try` block completes without raising an exception.

In [7]:
filename = "input.txt"

try:
    with open(filename, "r") as file:
        first_line = file.readline()
    value = float(first_line)
except FileNotFoundError:
    print(f"File not found: {filename}")
except ValueError:
    print("The file does not contain a valid numerical value in the first line.")
else:
    print(f"First value read successfully: {value}")

First value read successfully: 10.0


## 5. Comparison with Fortran I/O

Fortran uses a different model for file I/O. Files are connected to integer identifiers called **I/O units**. A typical file operation uses `OPEN`, `READ`, `WRITE`, and `CLOSE` statements.

```fortran
integer :: ios
real :: x

open(unit=10, file="input.txt", status="old", action="read", iostat=ios)
read(10, *, iostat=ios) x
close(10)
```

The variable `ios` stores the status of the I/O operation. A value equal to zero indicates success, while a non-zero value indicates that an error or end-of-file condition occurred.

Compared with Fortran, Python file I/O is more abstract: `open()` returns a file object, and reading or writing is performed through object methods such as `read()` and `write()`. The `with` statement also manages file closure automatically.

Fortran is more explicit: the programmer controls units, actions, status clauses, and formatting. This makes the syntax more verbose, but it also gives detailed control over formatted numerical output.

## 6. Pros and Cons from an HPC Perspective

| Operation | Python | Fortran |
| --- | --- | --- |
| File open and close | Compact syntax; `with` automatically closes the file | Uses I/O units and explicit `OPEN`/`CLOSE` statements |
| Reading data | High-level file methods such as `read()`, `readline()`, `readlines()` | `READ` statements associated with an I/O unit |
| Writing data | `write()` and `writelines()` write strings to a file | `WRITE` statements with strong formatting control |
| Error handling | Exceptions such as `FileNotFoundError` and `ValueError` | Status codes through clauses such as `IOSTAT` |
| Formatting | Flexible string formatting with f-strings or format methods | Dedicated `FORMAT` descriptors for numerical output |

Python is convenient for simple configuration files, small outputs, and quick prototyping. For large scientific datasets, pure text I/O in Python can become inefficient, so optimized libraries or binary formats are often preferred. Fortran provides structured and efficient I/O features that are widely used in scientific computing.

## 7. Summary

Python file I/O is based on file objects and methods such as `read()`, `write()`, and `writelines()`. The `with` statement is the preferred way to open files because it automatically closes them when the operation is complete.

Error handling is based on exceptions. With `try` and `except`, a program can react to missing files or invalid data without stopping unexpectedly.

Fortran uses a more explicit model based on I/O units, `OPEN`, `READ`, `WRITE`, `CLOSE`, and status variables such as `IOSTAT`. Python is generally more compact, while Fortran provides strong control over numerical formatting and traditional scientific I/O workflows.

## References

* Python documentation: [Input and Output](https://docs.python.org/3/tutorial/inputoutput.html)
* Python documentation: [Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html)
* Python documentation: [`open()` built-in function](https://docs.python.org/3/library/functions.html#open)
* Course material: [4_IO](https://moodle.unimore.it/mod/resource/view.php?id=443635).